In [ ]:
import pandas as pd
import numpy as np

from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
e003_metadata = pd.read_csv('e003_with_passage_one_redo_good.csv').set_index('sample')
e003_metadata['comm'].unique()
def get_both_dfs(fname):
    df1 = pd.read_csv(fname).drop(columns='Unnamed: 0').set_index('sample')
    df2 = pd.read_csv(fname.split('parent1_info.csv')[0] + 'parent2_info.csv').drop(columns='Unnamed: 0').set_index('sample')
    df2['conf_int'] = df2['boot_high']-df2['boot_low']
    
    df1['conf_int'] = df1['boot_high']-df1['boot_low']
    df_both = pd.concat([df1.rename(columns = {'boot_med':'boot_med1', 'boot_low':'boot_low1', 'boot_high':'boot_high1',
       'actual_med':'actual_med1', 'conf_int':'conf_int1'}),
                         df2.rename(columns = {'boot_med':'boot_med2', 'boot_low':'boot_low2', 'boot_high':'boot_high2',
       'actual_med':'actual_med2', 'conf_int':'conf_int2'})],axis=1)
    
    df_both['species'] = fname.split('/')[-2]
    df_both['fname'] = fname
   # print('-'.join(fname1.split('/')[-1].split('-')[:-1]))
   # parent_media = 
    
    df_both['subjects_measured'] = '-'.join(fname1.split('/')[-1].split('-')[:-1])
    df_both['in_measured'] = '-'.join(fname1.split('/')[-1].split('_')[:-2])
    df_both['total_shift'] = np.abs(1-(df_both['actual_med1'] + df_both['actual_med2']))
    df_both['total_shift12'] = np.abs(1-(df_both['boot_low1'] + df_both['boot_high2']))
    df_both['total_shift21'] = np.abs(1-(df_both['boot_low2'] + df_both['boot_high1']))
    df_both['total_shift_max'] = df_both['total_shift21']
    df_both.loc[df_both['total_shift_max']<df_both['total_shift12'],'total_shift_max'] = df_both.loc[df_both['total_shift_max']<df_both['total_shift12'],'total_shift12']

    return df_both
    

In [ ]:
folders = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/*/')
good_sp=[]
all_stuff = []
for folder in folders:
    species_id = folder.split('/')[-2]
    fnames = glob(f'{folder}/*_parent1_info.csv')
    plots = []
    for fname1 in fnames:
      #  if 'ACPP' in fname1: 
       #     print(fname1)
        #    continue
        df_both = get_both_dfs(fname1)
        df_both['boot_med1_shift'] = df_both['boot_med1']/(df_both['boot_med1']+df_both['boot_med2'])
       # df_both  = df_both.loc[df_both['total_shift'].abs()<.1,:]
       # print(len(df_both))
        df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
        df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                           e003_metadata.index.values),:]],
                                 axis=1).reset_index()
      #  print(len(df_both_meta))
        
        
        
        df_both_meta_good = df_both_meta.loc[df_both_meta['total_shift']<.1,:]
        df_both_meta_good['shift_from_0']= np.nan
        df_both_meta_good['species_id']=species_id
        for in_sample in df_both_meta_good['inoculumn_sample'].unique():
          # print(in_sample)
            if in_sample not in df_both_meta_good['sample'].values:
                continue
            in_value = df_both_meta_good.loc[df_both_meta_good['sample']==in_sample,'boot_med1'].values[0]
            df_both_meta_good.loc[df_both_meta_good['inoculumn_sample']==in_sample,'shift_from_0']=df_both_meta_good.loc[df_both_meta_good['inoculumn_sample']==in_sample,
                                        'boot_med1']-in_value
        df_both_meta_good['shift_from_one']=np.nan
        for mesocosm in df_both_meta_good['mesocosm'].unique():
            passages = df_both_meta_good.loc[df_both_meta_good['mesocosm']==mesocosm,'passage']
            if 3 not in passages.values:
                continue
            passage1=df_both_meta_good.loc[df_both_meta_good['mesocosm']==mesocosm,:]
          #  print(passages)
            passage1 = passage1.loc[passage1['passage']==3,'boot_med1'].values[0]
            df_both_meta_good.loc[df_both_meta_good['mesocosm']==mesocosm,'shift_from_one']=df_both_meta_good.loc[df_both_meta_good['mesocosm']==mesocosm,'boot_med1']-passage1
            
        all_stuff.append(df_both_meta_good)
        
            
       

In [ ]:
all_dfs = pd.concat(all_stuff)
all_dfs['species-type_mesocosm']=all_dfs['species_id']+'-'+all_dfs['type_mesocosm']
all_dfs_good=[]
for passage in [1,3,5,7]:
    all_dfs_passage = all_dfs.loc[all_dfs['passage']==passage,:]
    for sp_type_mesocosm in all_dfs_passage ['species-type_mesocosm'].unique():
        all_dfs_passage_sp_type=all_dfs_passage.loc[all_dfs_passage['species-type_mesocosm']==sp_type_mesocosm,:]
        med_value1 = np.mean(all_dfs_passage_sp_type['boot_med1'])#.median()
        med_value2 = np.mean(all_dfs_passage_sp_type['boot_med2'])#.median()
        all_dfs_passage_sp_type['med1']=med_value1 
        all_dfs_passage_sp_type['diff_med1']=med_value1-all_dfs_passage_sp_type['boot_med1']
        all_dfs_passage_sp_type['med2']=med_value2
        all_dfs_passage_sp_type['diff_med2']=med_value2-all_dfs_passage_sp_type['boot_med2']
        all_dfs_good.append(all_dfs_passage_sp_type)
        
        
        
        
        
df_err = pd.concat(all_dfs_good)

In [ ]:
p=iqplot.histogram(df_err,
                   q='diff_med1',rug=False,cats='passage')
bokeh.io.show(p)

In [ ]:
p=iqplot.ecdf(df_err, #.loc[df_err['diff_med1']!=0,:],
              q='diff_med1',cats='passage')
bokeh.io.show(p)

In [ ]:
p=iqplot.strip(df_err,#.loc[df_err['diff_med1']!=0,:],
               q='diff_med1',cats='passage',jitter=True)
bokeh.io.show(p)

In [ ]:
df_err['abs']=df_err['diff_med1'].abs()
p=iqplot.histogram(df_err,
                   q='abs',rug=False,cats='passage')
bokeh.io.show(p)
p=iqplot.ecdf(df_err,
                   q='abs',cats='passage')
bokeh.io.show(p)

p=iqplot.strip(df_err,jitter=True,
                   q='abs',cats='passage')
bokeh.io.show(p)

In [ ]:
#full_dfp7 = pd.read_csv('selection_coefficients_fitting2_v2.csv').set_index('species-mesocosm')

#full_dfp7 = full_dfp7.loc[~full_dfp7['parent_subjects'].isin(['AA-AC/PP',
 #                                                                   'AC/PP-AE','AC/PP-AF']),:]

#e003_metadata = pd.read_csv('e003_metadata_cultures_round2.csv').drop(columns='Unnamed: 0')
full_dfp7=all_dfs_only_one.copy()
full_dfp7['strain_winner'] = ''
full_dfp7.loc[full_dfp7['shift_from_one'] > 0, 'strain_winner'] =full_dfp7.loc[full_dfp7['shift_from_one'] > 0, 'parent_subjects'].transform(lambda x: x.split('-')[0])
full_dfp7.loc[full_dfp7['shift_from_one'] < 0, 'strain_winner'] =full_dfp7.loc[full_dfp7['shift_from_one'] < 0, 'parent_subjects'].transform(lambda x: x.split('-')[1])
full_dfp7['p7_s']=full_dfp7['shift_from_one']
full_dfp7['type_meso']=full_dfp7['type_mesocosm']
full_dfp7 = full_dfp7.loc[~full_dfp7['parent_subjects'].isin(['AA-AC/PP',
                                                                    'AC/PP-AE','AC/PP-AF']),:]
full_dfp7['species']

In [ ]:

#full_dfp7_good = full_dfp7_good.loc[full_dfp7_good['p7_freq'] > 0,:]
#full_dfp7_good = full_dfp7_good.loc[full_dfp7_good['p7_freq'] < 1,:]

groups = [['AA-AC/PP', 'AA-AE', 'AC/PP-AE'], ['AA-AC/PP', 'AA-AF', 'AC/PP-AF'], 
          ['AA-AE', 'AA-AF', 'AE-AF'], ['AC/PP-AE', 'AC/PP-AF','AE-AF' ],]

media_pairs = ['mBHI-mBHI', 'mBHI-mGAM', 'mGAM-mGAM', 'mGAM-mBHI']

candidates = []
for sp in full_dfp7['species_id'].unique():
    full_dfp7_sp = full_dfp7.loc[full_dfp7['species_id'] == sp,:]
    for media_p in media_pairs:
        parent_media, media = media_p.split('-')
        full_dfp7_good = full_dfp7_sp.loc[full_dfp7_sp['media'] == media,:]
        full_dfp7_good = full_dfp7_good.loc[full_dfp7_good['parent_media'] == parent_media,:]
        for gr in groups:
            full_dfp7_goodgr = full_dfp7_good.loc[full_dfp7_good['parent_subjects'].isin(gr),:].sort_values(by='parent_subjects')
            id_columns = ['parent_subjects', 'media', 'parent_media', 'species_id','type_meso','species']
            
            full_dfp7_goodgr = full_dfp7_goodgr[id_columns + ['p7_s']].groupby(id_columns).median().reset_index()
            full_dfp7_goodgr = full_dfp7_goodgr.loc[~full_dfp7_goodgr['p7_s'].isna(),:]
            full_dfp7_goodgr['strain_winner'] = ''
            full_dfp7_goodgr.loc[full_dfp7_goodgr['p7_s'] > 0, 'strain_winner'] = \
                full_dfp7_goodgr.loc[full_dfp7_goodgr['p7_s'] > 0, 'parent_subjects'].transform(lambda x: x.split('-')[0])
            full_dfp7_goodgr.loc[full_dfp7_goodgr['p7_s'] < 0, 'strain_winner'] = \
                full_dfp7_goodgr.loc[full_dfp7_goodgr['p7_s'] < 0, 'parent_subjects'].transform(lambda x: x.split('-')[1])
            

            #print(gr)
            subs ='-'.join(np.unique('-'.join(gr).split('-')))
            full_dfp7_goodgr['trio'] = f'{sp}-{subs}-{media_p}'
         
            
            #print(full_dfp7_goodgr[['strain_freq','p7_s', 'opp_p7_s']])
            if len(full_dfp7_goodgr) == 3:
                sp_plot = full_dfp7_goodgr['species'].values[0].split('s__')[-1]
                full_dfp7_goodgr['trio_plot'] = f'{sp_plot}-{subs}-{media_p}'
                full_dfp7_goodgr['non_transitive'] = ''
                if len(full_dfp7_goodgr['strain_winner'].unique()) >2:
                    full_dfp7_goodgr['non_transitive'] = '**'
                    
                candidates.append(full_dfp7_goodgr)
                
                    
                  #  print(full_dfp7_goodgr[['trio','type_meso', 'p7_s', 'strain_winner', 'species_id','species' ]])


        
    

In [ ]:
all_cands = pd.concat(candidates)
len(all_cands)

In [ ]:
full_dfp7['total_shift'].max()

In [ ]:
all_cands = pd.concat(candidates)
all_cands['non_transitive_trio_plot'] = all_cands['non_transitive'] + ' ' + all_cands['trio_plot']
all_cands['count']=1.

bars = hv.Bars(all_cands.sort_values(by='trio_plot',ascending=False) , kdims=['non_transitive_trio_plot','strain_winner'],
               vdims = ['count'])

bars.opts(width=700,height = 550, ).opts(stacked=True,  ylabel='', xlabel='',
                                   xaxis=None,
                                         invert_axes=True,#xrotation = 90,
                                         cmap = bokeh.palettes.Set3[12])

bars.opts(legend_position='left',)#4*6

In [ ]:
len(full_dfp7)

In [ ]:
full_dfp7['species-type_meso'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['type_meso']
full_dfp7['species-type_meso']

In [ ]:
full_dfp7.loc[full_dfp7['species-type_meso'].isin(['102528-AA-AE-mBHI-mGAM',
                                                             '102528-AA-AF-mBHI-mGAM',
                                                             '102528-AE-AF-mBHI-mGAM',]),'p7_s']
                                                             
AA>>AF, AF>AE, AE>AA 

In [ ]:
full_dfp7.loc[full_dfp7['species-type_meso'].isin(['100146-AA-AE-mGAM-mBHI',
                                                             '100146-AA-AF-mGAM-mBHI',
                                                             '100146-AE-AF-mGAM-mBHI',]),'p7_s']
   #good 

In [ ]:
len(all_cands.loc[all_cands['non_transitive']=='**',:].groupby(['trio']).sum())


In [ ]:
len(all_cands)